# TikTok Video Engagement Prediction — Day 30 Views
**WeCloudData DS Bootcamp — In-Class Competition**

Predicts cumulative views at Day 30 using only metadata, creator stats,
and daily engagement from days 0-5 (no data leakage).

**Approach:** Predict the growth ratio (target / day-5 views) in log-space
using LightGBM with a Huber loss, then convert back to raw views.

**Final CV RMSE:** ~59,197 | **Kaggle Public Score:** 86,303.49


In [8]:
!pip install -q lightgbm
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import GroupKFold


## 1. Load Data


In [9]:
from google.colab import files
import zipfile, os

up = files.upload()  # upload predictive-modelling-ds.zip
os.makedirs("kaggle_data", exist_ok=True)
for f in up:
    zipfile.ZipFile(f).extractall("kaggle_data")

train = pd.read_csv("kaggle_data/train_videos.csv")
test  = pd.read_csv("kaggle_data/test_videos.csv")
eng   = pd.read_csv("kaggle_data/engagement_daily.csv")
cre   = pd.read_csv("kaggle_data/creators_daily.csv")
cre["date"] = pd.to_datetime(cre["date"])
cre = cre.sort_values("date")

print("train:", train.shape, "| test:", test.shape)
print("engagement:", eng.shape, "| creators:", cre.shape)


Saving predictive-modelling-ds.zip to predictive-modelling-ds (1).zip
train: (12000, 27) | test: (3001, 26)
engagement: (79489, 10) | creators: (252166, 7)


## 2. Feature Engineering
Build day 0-5 engagement features (forward-filled), ratios, growth signals,
and creator stats as of day 5 (no leakage past day 5).


In [10]:
metrics = ["play_count", "like_count", "comment_count", "share_count",
           "collect_count", "download_count", "whatsapp_share_count"]

def pivot_engagement(video_ids):
    e = eng[eng.video_id.isin(video_ids) & (eng.days_since_post <= 5)]
    w = e.pivot(index="video_id", columns="days_since_post", values=metrics)
    w.columns = [f"{m}_d{d}" for m, d in w.columns]
    for m in metrics:
        cols = [f"{m}_d{i}" for i in range(1, 6) if f"{m}_d{i}" in w.columns]
        w[cols] = w[cols].ffill(axis=1)
    return w.reset_index()

def build_features(meta, video_ids):
    d = meta.merge(pivot_engagement(video_ids), on="video_id", how="left")
    d["create_time"] = pd.to_datetime(d["create_time"])
    d["hour"] = d.create_time.dt.hour
    d["dow"]  = d.create_time.dt.dayofweek

    d["like_ratio"]  = d.like_count_d5  / (d.play_count_d5 + 1)
    d["share_ratio"] = d.share_count_d5 / (d.play_count_d5 + 1)
    d["growth_5_1"]  = d.play_count_d5  / (d.play_count_d1 + 1)
    d["growth_5_4"]  = d.play_count_d5  / (d.play_count_d4 + 1)

    d["date5"] = pd.to_datetime(d["create_date"]) + pd.Timedelta(days=5)
    d = d.sort_values("date5")
    d = pd.merge_asof(d, cre, left_on="date5", right_on="date",
                       by="author_id", direction="backward", suffixes=("", "_cre"))
    d["log_followers"]       = np.log1p(d.follower_count)
    d["plays_per_follower"]  = d.play_count_d5 / (d.follower_count + 1)
    d["like_per_follower"]   = d.total_favorited / (d.follower_count + 1)
    d["avg_likes_per_video"] = d.total_favorited / (d.video_count + 1)

    d["enterprise_verified"] = d["enterprise_verified"].astype("category")
    return d.sort_values("video_id").reset_index(drop=True)

cat_cols = ["ratio", "desc_language", "topic", "is_ads",
            "music_selected_from", "music_author", "enterprise_verified"]
drop_cols = ["video_id", "author_id", "create_time", "create_date",
             "music_id", "music_owner_id", "date5", "date"]

d_train = build_features(train, train.video_id)
d_test  = build_features(test,  test.video_id)

for c in cat_cols:
    d_train[c] = d_train[c].astype("category")
    d_test[c]  = d_test[c].astype("category")
    d_test[c]  = d_test[c].cat.set_categories(d_train[c].cat.categories)

X_train = d_train.drop(columns=drop_cols + ["target_day30_views"])
X_test  = d_test.drop(columns=drop_cols)[X_train.columns]

y_ratio = d_train["target_day30_views"] / (d_train["play_count_d5"].fillna(0) + 1)
y_log   = np.log(y_ratio + 1e-6)
base_train = d_train["play_count_d5"].fillna(0).values + 1
base_test  = d_test["play_count_d5"].fillna(0).values + 1
groups = d_train["author_id"]

print("Features:", X_train.shape[1])
print("NaN check - train:", d_train.play_count_d5.isna().sum(),
      "| test:", d_test.play_count_d5.isna().sum())


Features: 77
NaN check - train: 1 | test: 1


## 3. Model Training & Cross-Validation
Predict log(growth ratio) with LightGBM (Huber loss, alpha=0.2),
validated with GroupKFold on author_id to prevent creator leakage.


In [11]:
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(d_train))
for tr, va in gkf.split(X_train, y_log, groups):
    m = lgb.LGBMRegressor(objective="huber", alpha=0.2, n_estimators=1000,
                          learning_rate=0.03, num_leaves=31, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          random_state=42, verbose=-1)
    m.fit(X_train.iloc[tr], y_log.iloc[tr], eval_set=[(X_train.iloc[va], y_log.iloc[va])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    oof[va] = np.exp(m.predict(X_train.iloc[va])) * base_train[va]

cv_rmse = np.sqrt(np.mean((oof - d_train.target_day30_views.values) ** 2))
print("CV RMSE:", round(cv_rmse))


CV RMSE: 59197


## 4. Final Model & Test Predictions
Train on all training data (5 seeds averaged) and predict Day-30 views for the test set.


In [12]:
preds = []
for s in range(5):
    m = lgb.LGBMRegressor(objective="huber", alpha=0.2, n_estimators=400,
                          learning_rate=0.03, num_leaves=31, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          random_state=s, verbose=-1)
    m.fit(X_train, y_log)
    preds.append(np.exp(m.predict(X_test)) * base_test)

final_pred = np.clip(np.mean(preds, axis=0), 0, None)
sub = pd.DataFrame({"video_id": d_test.video_id, "target_day30_views": final_pred})
sub.to_csv("submission.csv", index=False)
print(sub.head())
print("Rows:", len(sub))


              video_id  target_day30_views
0  7384177529092951338        14067.539098
1  7384193851700956447         4938.605413
2  7384194290215292190          389.357709
3  7384194754822655278          332.449640
4  7384197193470676266         8669.766889
Rows: 3001


## 5. Summary

| Model | RMSE |
|---|---|
| Mean baseline | 324,033 |
| Day-5 × median growth | 145,815 |
| LightGBM (log-target) | 173,415 |
| LightGBM (log-ratio, L2) | 128,291 |
| **LightGBM (log-ratio, Huber α=0.2)** | **59,197 (CV)** |

**Kaggle Public Score:** 86,303.49

**Key findings:**
- The target follows a heavy power-law distribution; a few viral videos
  dominate the RMSE (~70% of squared error from ~20 videos).
- Predicting the *growth ratio* (target / day-5 views) in log-space,
  rather than the raw target, was the single biggest improvement.
- Huber loss (less sensitive to outliers than L2) further improved results.
- Creator features (followers, avg likes/video) gave a small, consistent
  improvement on the log-scale metric, even though it didn't always show
  on raw RMSE due to noise from the top outlier videos.
- GroupKFold on `author_id` was used throughout to prevent the same
  creator appearing in both train and validation.


In [13]:
from google.colab import files
files.download("submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>